In [ ]:
# Starter for Version A (paste into a notebook cell)
import heapq
import time
import tracemalloc
from collections import defaultdict
import random

class CityGraph:
    def __init__(self):
        self.adj = defaultdict(list)  # node -> list of (neighbor, weight)

    def add_edge(self, u, v, w):
        self.adj[u].append((v,w))
        self.adj[v].append((u,w))

    def dijkstra(self, src, target):
        # returns distance (float('inf') if unreachable)
        pq = [(0, src)]
        dist = {src: 0}
        while pq:
            d,u = heapq.heappop(pq)
            if u == target:
                return d
            if d != dist.get(u, float('inf')):
                continue
            for v,w in self.adj[u]:
                nd = d + w
                if nd < dist.get(v, float('inf')):
                    dist[v] = nd
                    heapq.heappush(pq, (nd, v))
        return float('inf')

class Ambulance:
    def __init__(self, aid, location):
        self.aid = aid
        self.location = location
        self.available = True

class DispatchSystem:
    def __init__(self, graph, ambulances):
        self.graph = graph
        self.ambulances = ambulances

    def assign_ambulance(self, incident_loc, urgency=1):
        heap = []
        for amb in self.ambulances:
            if not amb.available:
                continue
            dist = self.graph.dijkstra(amb.location, incident_loc)
            # priority tuple: (-urgency, distance, id) -> higher urgency first
            heapq.heappush(heap, (-urgency, dist, amb.aid, amb))
        if not heap:
            return None
        _, dist, _, amb = heapq.heappop(heap)
        amb.available = False
        return amb, dist

# quick random test generator
def make_random_graph(n_nodes=20, edge_prob=0.2, max_w=10):
    G = CityGraph()
    for i in range(n_nodes):
        for j in range(i+1, n_nodes):
            if random.random() < edge_prob:
                w = random.randint(1, max_w)
                G.add_edge(i, j, w)
    return G

# small simulation + benchmarking
random.seed(0)
G = make_random_graph(30, edge_prob=0.12)
ambs = [Ambulance(i, random.randrange(0,30)) for i in range(5)]
ds = DispatchSystem(G, ambs)

# benchmark single assignment (time + memory)
tracemalloc.start()
t0 = time.perf_counter()
assigned = ds.assign_ambulance(incident_loc=10, urgency=3)
t1 = time.perf_counter()
cur, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("assigned:", (assigned[0].aid if assigned else None), "dist:", (assigned[1] if assigned else None))
print("time_s:", t1-t0, "mem_cur_kb:", cur/1024, "mem_peak_kb:", peak/1024)
